In [1]:
# 批量点云指标：CD / F1@τ / Completeness / Accuracy / RMSE + ICP Rotation/Translation
import sys
import subprocess
from pathlib import Path
import numpy as np
from scipy.spatial import cKDTree
import trimesh

# ===== 输入目录（每个目录 9 个 glb） =====
GT_DIR = "/home/viplab/liukejia/project/result/reconstruct_gt_scale"
PRED_DIR = "/home/viplab/liukejia/project/result/reconstruct_pred_scale"


In [2]:

# 采样与阈值设置
MAX_POINTS = 120000
F1_THRESHOLDS = [0.2, 0.5]
ICP_MAX_ITER = 50
ICP_TOL = 1e-6
ICP_SAMPLE_SIZE = 8000

def load_points_from_glb(glb_path: str, max_points: int = 120000, seed: int = 0):
    obj = trimesh.load(glb_path, force="scene")

    if isinstance(obj, trimesh.Trimesh):
        pts = np.asarray(obj.vertices, dtype=np.float64)
    else:
        # Scene -> 合并成单个 mesh（含变换）
        merged = obj.dump(concatenate=True)
        if isinstance(merged, (list, tuple)):
            merged = merged[0] if len(merged) > 0 else None
        if merged is None or not hasattr(merged, "vertices"):
            raise ValueError(f"无法从场景解析顶点: {glb_path}")
        pts = np.asarray(merged.vertices, dtype=np.float64)

    if pts.shape[0] == 0:
        raise ValueError(f"空点云: {glb_path}")

    if max_points is not None and pts.shape[0] > max_points:
        rng = np.random.default_rng(seed)
        idx = rng.choice(pts.shape[0], size=max_points, replace=False)
        pts = pts[idx]
    return pts

def nearest_distances(src_pts: np.ndarray, dst_pts: np.ndarray):
    tree = cKDTree(dst_pts)
    dists, _ = tree.query(src_pts, k=1, workers=-1)
    return dists.astype(np.float64)

def chamfer_distance_l2_squared(p_rec: np.ndarray, p_gt: np.ndarray):
    d_rec = nearest_distances(p_rec, p_gt)
    d_gt = nearest_distances(p_gt, p_rec)
    cd = np.mean(d_rec ** 2) + np.mean(d_gt ** 2)
    return float(cd), d_rec, d_gt

def f1_at_threshold(d_rec: np.ndarray, d_gt: np.ndarray, tau: float):
    precision = float(np.mean(d_rec < tau))
    recall = float(np.mean(d_gt < tau))
    f1 = 0.0 if (precision + recall) == 0 else 2.0 * precision * recall / (precision + recall)
    return precision, recall, f1

def pointcloud_metrics(p_rec: np.ndarray, p_gt: np.ndarray, thresholds=(0.2, 0.5)):
    cd, d_rec, d_gt = chamfer_distance_l2_squared(p_rec, p_gt)
    accuracy = float(np.mean(d_rec))
    completeness = float(np.mean(d_gt))
    rmse_nn = float(np.sqrt((np.mean(d_rec ** 2) + np.mean(d_gt ** 2)) / 2.0))

    out = {
        "cd_l2_squared": cd,
        "accuracy": accuracy,
        "completeness": completeness,
        "rmse_nn": rmse_nn,
        "num_rec": int(p_rec.shape[0]),
        "num_gt": int(p_gt.shape[0]),
        "f1": {},
    }
    for tau in thresholds:
        p, r, f1 = f1_at_threshold(d_rec, d_gt, tau=float(tau))
        out["f1"][float(tau)] = {"precision": p, "recall": r, "f1": f1}
    return out

def best_fit_transform(A: np.ndarray, B: np.ndarray):
    """求解 A->B 的刚体变换（Kabsch）"""
    centroid_A = np.mean(A, axis=0)
    centroid_B = np.mean(B, axis=0)

    AA = A - centroid_A
    BB = B - centroid_B

    H = AA.T @ BB
    U, _, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T

    # 处理反射
    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T

    t = centroid_B - R @ centroid_A
    return R, t

def icp_point_to_point(src: np.ndarray, dst: np.ndarray, max_iter=50, tol=1e-6, sample_size=8000, seed=0):
    """返回将 src 对齐到 dst 的刚体变换 R,t（src_aligned = R@src + t）"""
    rng = np.random.default_rng(seed)
    tree = cKDTree(dst)

    R_total = np.eye(3, dtype=np.float64)
    t_total = np.zeros(3, dtype=np.float64)

    prev_error = np.inf
    src0 = src.astype(np.float64, copy=False)

    for _ in range(max_iter):
        src_trans = (R_total @ src0.T).T + t_total

        if src_trans.shape[0] > sample_size:
            idx = rng.choice(src_trans.shape[0], size=sample_size, replace=False)
            src_trans_s = src_trans[idx]
        else:
            src_trans_s = src_trans

        dists, nn_idx = tree.query(src_trans_s, k=1, workers=-1)
        dst_match = dst[nn_idx]

        dR, dt = best_fit_transform(src_trans_s, dst_match)

        # 累积变换
        R_total = dR @ R_total
        t_total = dR @ t_total + dt

        mean_error = float(np.mean(dists))
        if abs(prev_error - mean_error) < tol:
            break
        prev_error = mean_error

    return R_total, t_total

def rotation_angle_deg_from_R(R: np.ndarray):
    tr = np.trace(R)
    cos_theta = np.clip((tr - 1.0) / 2.0, -1.0, 1.0)
    theta = np.arccos(cos_theta)
    return float(np.degrees(theta))

def evaluate_one_pair(pred_glb: str, gt_glb: str, seed_base: int):
    p_pred = load_points_from_glb(pred_glb, max_points=MAX_POINTS, seed=seed_base)
    p_gt = load_points_from_glb(gt_glb, max_points=MAX_POINTS, seed=seed_base + 1)

    # 原始指标
    m_raw = pointcloud_metrics(p_pred, p_gt, thresholds=F1_THRESHOLDS)

    # ICP 刚体对齐
    R, t = icp_point_to_point(
        p_pred, p_gt,
        max_iter=ICP_MAX_ITER,
        tol=ICP_TOL,
        sample_size=ICP_SAMPLE_SIZE,
        seed=seed_base + 2,
    )
    rot_err_deg = rotation_angle_deg_from_R(R)
    trans_err = float(np.linalg.norm(t))

    # 对齐后指标（可选，通常更公平）
    p_pred_aligned = (R @ p_pred.T).T + t
    m_icp = pointcloud_metrics(p_pred_aligned, p_gt, thresholds=F1_THRESHOLDS)

    return {
        "raw": m_raw,
        "icp": m_icp,
        "icp_rotation_error_deg": rot_err_deg,
        "icp_translation_error": trans_err,
    }

# ===== 批量匹配并评估 =====
gt_files = sorted(Path(GT_DIR).glob("*.glb"))
pred_files = sorted(Path(PRED_DIR).glob("*.glb"))

if len(gt_files) == 0 or len(pred_files) == 0:
    raise FileNotFoundError("GT_DIR 或 PRED_DIR 下没有找到 .glb 文件")

gt_map = {p.name: str(p) for p in gt_files}
pred_map = {p.name: str(p) for p in pred_files}
common_names = sorted(set(gt_map.keys()) & set(pred_map.keys()))

if len(common_names) == 0:
    raise ValueError("两个目录没有同名 glb，无法自动配对")

missing_in_pred = sorted(set(gt_map.keys()) - set(pred_map.keys()))
missing_in_gt = sorted(set(pred_map.keys()) - set(gt_map.keys()))

print(f"GT 文件数: {len(gt_files)}, PRED 文件数: {len(pred_files)}, 成功配对: {len(common_names)}")
if missing_in_pred:
    print("仅 GT 有: ", missing_in_pred)
if missing_in_gt:
    print("仅 PRED 有: ", missing_in_gt)

rows = []
for i, name in enumerate(common_names):
    result = evaluate_one_pair(pred_map[name], gt_map[name], seed_base=1000 + 10 * i)
    rows.append((name, result))

# ===== 打印逐文件结果 =====
for name, r in rows:
    raw = r["raw"]
    icp = r["icp"]
    print("=" * 80)
    print(f"[{name}]")
    print(f"RAW   | CD: {raw['cd_l2_squared']:.6f}, Acc: {raw['accuracy']:.6f}, Comp: {raw['completeness']:.6f}, RMSE: {raw['rmse_nn']:.6f}")
    for tau in F1_THRESHOLDS:
        s = raw["f1"][float(tau)]
        print(f"RAW   | F1@{tau}: P={s['precision']:.4f}, R={s['recall']:.4f}, F1={s['f1']:.4f}")

    print(f"ICP   | RotErr(deg): {r['icp_rotation_error_deg']:.6f}, TransErr: {r['icp_translation_error']:.6f}")
    print(f"ICP   | CD: {icp['cd_l2_squared']:.6f}, Acc: {icp['accuracy']:.6f}, Comp: {icp['completeness']:.6f}, RMSE: {icp['rmse_nn']:.6f}")
    for tau in F1_THRESHOLDS:
        s = icp["f1"][float(tau)]
        print(f"ICP   | F1@{tau}: P={s['precision']:.4f}, R={s['recall']:.4f}, F1={s['f1']:.4f}")

# ===== 汇总平均 =====
def avg(values):
    return float(np.mean(values)) if len(values) else float("nan")

avg_raw_cd = avg([r["raw"]["cd_l2_squared"] for _, r in rows])
avg_raw_acc = avg([r["raw"]["accuracy"] for _, r in rows])
avg_raw_comp = avg([r["raw"]["completeness"] for _, r in rows])
avg_raw_rmse = avg([r["raw"]["rmse_nn"] for _, r in rows])

avg_icp_rot = avg([r["icp_rotation_error_deg"] for _, r in rows])
avg_icp_trans = avg([r["icp_translation_error"] for _, r in rows])
avg_icp_cd = avg([r["icp"]["cd_l2_squared"] for _, r in rows])
avg_icp_acc = avg([r["icp"]["accuracy"] for _, r in rows])
avg_icp_comp = avg([r["icp"]["completeness"] for _, r in rows])
avg_icp_rmse = avg([r["icp"]["rmse_nn"] for _, r in rows])

print("\n" + "#" * 80)
print("平均结果 (ALL PAIRED FILES)")
print(f"RAW   | CD: {avg_raw_cd:.6f}, Acc: {avg_raw_acc:.6f}, Comp: {avg_raw_comp:.6f}, RMSE: {avg_raw_rmse:.6f}")

for tau in F1_THRESHOLDS:
    vals = [r["raw"]["f1"][float(tau)]["f1"] for _, r in rows]
    print(f"RAW   | Mean F1@{tau}: {avg(vals):.4f}")

print(f"ICP   | Mean RotErr(deg): {avg_icp_rot:.6f}, Mean TransErr: {avg_icp_trans:.6f}")
print(f"ICP   | CD: {avg_icp_cd:.6f}, Acc: {avg_icp_acc:.6f}, Comp: {avg_icp_comp:.6f}, RMSE: {avg_icp_rmse:.6f}")
for tau in F1_THRESHOLDS:
    vals = [r["icp"]["f1"][float(tau)]["f1"] for _, r in rows]
    print(f"ICP   | Mean F1@{tau}: {avg(vals):.4f}")

GT 文件数: 9, PRED 文件数: 9, 成功配对: 9


/tmp/ipykernel_1598941/19175499.py:15: DeprecationWarning: `Scene.dump(concatenate=True)` DEPRECATED FOR REMOVAL APRIL 2025: replace with `Scene.to_geometry()`
  merged = obj.dump(concatenate=True)


[num123_-800_100_0_0007.glb]
RAW   | CD: 5.776851, Acc: 1.600470, Comp: 1.555255, RMSE: 1.699537
RAW   | F1@0.2: P=0.1429, R=0.1429, F1=0.1429
RAW   | F1@0.5: P=0.1429, R=0.1429, F1=0.1429
ICP   | RotErr(deg): 0.002540, TransErr: 1.408674
ICP   | CD: 2.185943, Acc: 1.013306, Comp: 0.965924, RMSE: 1.045453
ICP   | F1@0.2: P=0.0000, R=0.0000, F1=0.0000
ICP   | F1@0.5: P=0.0000, R=0.0000, F1=0.0000
[num123_-800_150_0_0008.glb]
RAW   | CD: 0.007441, Acc: 0.056677, Comp: 0.056677, RMSE: 0.060997
RAW   | F1@0.2: P=1.0000, R=1.0000, F1=1.0000
RAW   | F1@0.5: P=1.0000, R=1.0000, F1=1.0000
ICP   | RotErr(deg): 0.000002, TransErr: 0.044070
ICP   | CD: 0.003558, Acc: 0.040879, Comp: 0.040879, RMSE: 0.042178
ICP   | F1@0.2: P=1.0000, R=1.0000, F1=1.0000
ICP   | F1@0.5: P=1.0000, R=1.0000, F1=1.0000
[num123_-800_200_0_0009.glb]
RAW   | CD: 0.244166, Acc: 0.324664, Comp: 0.324664, RMSE: 0.349404
RAW   | F1@0.2: P=0.1429, R=0.1429, F1=0.1429
RAW   | F1@0.5: P=1.0000, R=1.0000, F1=1.0000
ICP   | RotEr